# 26 - Team encoder on teams never seen (experiment 13)

Ideas R1 and R3 in `EXPERIMENTS.md`. The team features give +0.90 points when teams repeat, but much of that signal is
**rarity** (a P1 team that appears only once wins about 40% of its battles), and we have no clean measure of what
happens on a private set with new teams. The previous stress test hid both sides at once and also erased what is still
known about a new team. Two changes are tested together, once, with no further iteration:

- **R1, species pairs** (`pairs=True`): a new team made of known species still has a pair score;
- **R3, per-side masking** (`per_side=True`): `train_unseen` is drawn separately for P1 and P2, because a new P1 team
  usually has a known P2 side, and after joint masking the rows with n1 = 0 and n2 > 0 still win only 36%.

**Measure on unseen teams** (`unseen_cv_scores`): after fitting, the encoder's pool is rebuilt without the battles that
contain a validation team, in either role. Two scenarios: **P1 only** (validation P1 teams removed) and **both** (also
the fully revealed validation P2 teams). Species stay known. Limits: the model still trains on those battles, and if
the private set also had new species, the pair scores would not help.

**Decision rule (fixed before running)**, LightGBM of experiment 04, paired on 15 folds (seeds 42 / 7 / 123 × 5), against
the current encoder of submission 24 (`train_unseen=0.3`):
- clean CV: no more than **0.1 points** lower;
- unseen teams, in **both** scenarios: at least **+0.3 points** and at least **11 of 15 folds** improved;
- permuted team keys: less than +0.2 points over the model without team features.

R1 + R3 is the candidate. R1 alone and R3 alone are there to see which one does the work. If no configuration passes,
submissions 22-24 stay as they are and feature work stops.

## 0. Setup

In [1]:
import os
import sys
import json
import time
import warnings
from pathlib import Path

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features import team_key_features
from src.models import get_model_specs, with_params
from src.team_strength import KEY_P1, KEY_P2, TeamStrengthEncoder, unseen_cv_scores, with_team_strength
from src.validation import REPEAT_SEEDS, SEED, cv_scores

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
PROC = ROOT / "data" / "processed"
OUT = PROC / "models"
OUT.mkdir(parents=True, exist_ok=True)

team, states = [pd.read_pickle(PROC / f"{name}_train.pkl") for name in ("team", "states")]
ref = pd.read_pickle(PROC / "features_train.pkl")             # v1 features (notebook 03)
with open(PROC / "feature_groups.json") as f:
    groups = json.load(f)
y = ref.pop("player_won")
NUM = [c for c in ref.columns if c not in groups["species_cat"]]
KEYS = [KEY_P1, KEY_P2]
X = pd.concat([ref[NUM], team_key_features(team, states).reindex(ref.index)], axis=1)
assert X[KEYS].notna().all().all()
print(f"battles: {len(X):,} | features: {len(NUM)} + team keys")

specs = get_model_specs()
LGBM = with_params(specs["lightgbm"], {
    "model__colsample_bytree": 0.6286197472598067, "model__learning_rate": 0.006552992991573526,
    "model__min_child_samples": 106, "model__n_estimators": 1424, "model__num_leaves": 12,
    "model__reg_alpha": 0.6070874110546393, "model__reg_lambda": 2.7129089558470034,
    "model__subsample": 0.6629794526009424})

CURRENT = {"train_unseen": 0.3}                         # encoder of submissions 22 and 24
ENCODERS = {
    "current": CURRENT,
    "R3 per side": {**CURRENT, "per_side": True},
    "R1 pairs": {**CURRENT, "pairs": True},
    "R1 + R3": {**CURRENT, "pairs": True, "per_side": True},
    "encoder of submission 23": {"counts": False, "train_unseen": 0.5},
}

battles: 10,000 | features: 155 + team keys


## 1. Checks on the encoder

- **Own label:** the labels of all the battles of one P1 team are flipped: the pair scores of those battles must not
  change, because each row is encoded from a pool without its own teams.
- **Defaults unchanged:** an encoder with the current parameters must not produce pair columns, so submissions 22-24
  are not affected by the new code.

In [2]:
Xk = X[NUM + KEYS]
encoder = TeamStrengthEncoder(**ENCODERS["R1 + R3"])
base_enc = encoder.fit_transform(Xk, y)

team_id = X[KEY_P1].value_counts().index[0]
rows = X[KEY_P1] == team_id
flipped = y.where(~rows, 1 - y)
flip_enc = TeamStrengthEncoder(**ENCODERS["R1 + R3"]).fit_transform(Xk, flipped)
change = (base_enc.loc[rows, ["pr1", "pr2"]] - flip_enc.loc[rows, ["pr1", "pr2"]]).abs().max().max()
print(f"team flipped: {int(rows.sum())} battles | largest change in their pair scores: {change:.2e}")
assert change == 0.0, "a row sees its own label through the pair scores"

current_cols = TeamStrengthEncoder(**CURRENT).fit_transform(Xk, y).columns
assert not {"pr1", "pr2", "diff_pr"} & set(current_cols)
out = ["te_p1", "te_p2", "te_n_p1", "te_n_p2", "diff_te", "pr1", "pr2", "diff_pr"]
pd.DataFrame({"mean": base_enc[out].mean(), "std": base_enc[out].std(), "corr with winner": base_enc[out].corrwith(y)}).round(3)

team flipped: 195 battles | largest change in their pair scores: 0.00e+00


,mean,std,corr with winner
te_p1,0.503,0.052,0.106
te_p2,0.503,0.051,-0.071
te_n_p1,1.404,1.802,0.046
te_n_p2,1.821,2.178,-0.057
diff_te,0.000,0.073,0.125
pr1,-0.019,0.019,0.084
pr2,-0.017,0.021,-0.066
diff_pr,-0.002,0.027,0.108


## 2. Paired CV: clean and with teams never seen

Every configuration on the same 15 folds. `no team features` is the model without the encoder: it is the reference
that does not depend on the team pool at all.

In [3]:
def repeated(fn):
    return np.concatenate([fn(s) for s in REPEAT_SEEDS])

t0 = time.time()
no_team = repeated(lambda s: cv_scores(LGBM, X[NUM], y, seed=s))
scores = {}
for name, params in ENCODERS.items():
    model = with_team_strength(LGBM, **params)
    scores[name] = {
        "clean": repeated(lambda s: cv_scores(model, X[NUM + KEYS], y, seed=s)),
        "unseen P1": repeated(lambda s: unseen_cv_scores(model, X[NUM + KEYS], y, side="p1", seed=s)),
        "unseen both": repeated(lambda s: unseen_cv_scores(model, X[NUM + KEYS], y, side="both", seed=s)),
    }
    print(f"{name:26s} clean {scores[name]['clean'].mean():.4f} | unseen P1 {scores[name]['unseen P1'].mean():.4f} | "
          f"unseen both {scores[name]['unseen both'].mean():.4f} ({time.time() - t0:.0f}s)")
print(f"{'no team features':26s} {no_team.mean():.4f}")

current                    clean 0.6867 | unseen P1 0.6745 | unseen both 0.6743 (52s)
R3 per side                clean 0.6845 | unseen P1 0.6760 | unseen both 0.6742 (91s)
R1 pairs                   clean 0.6900 | unseen P1 0.6763 | unseen both 0.6752 (137s)
R1 + R3                    clean 0.6888 | unseen P1 0.6788 | unseen both 0.6771 (185s)
encoder of submission 23   clean 0.6846 | unseen P1 0.6769 | unseen both 0.6767 (226s)
no team features           0.6779


In [4]:
rows = []
for name in ENCODERS:
    row = {"encoder": name}
    for mode in ("clean", "unseen P1", "unseen both"):
        d_cur = scores[name][mode] - scores["current"][mode]
        d_base = scores[name][mode] - no_team
        row[f"{mode}: acc"] = scores[name][mode].mean()
        row[f"{mode}: vs current"] = d_cur.mean() * 100
        row[f"{mode}: folds"] = int((d_cur > 0).sum())
        row[f"{mode}: vs no team"] = d_base.mean() * 100
    rows.append(row)
table = pd.DataFrame(rows).set_index("encoder")
table.to_csv(OUT / "encoder_unseen_teams.csv")
table.T.round(4)

encoder,current,R3 per side,R1 pairs,R1 + R3,encoder of submission 23
clean: acc,0.6867,0.6845,0.6900,0.6888,0.6846
clean: vs current,0.0000,-0.2200,0.3300,0.2133,-0.2100
clean: folds,0.0000,2.0000,10.0000,8.0000,3.0000
clean: vs no team,0.8767,0.6567,1.2067,1.0900,0.6667
unseen P1: acc,0.6745,0.6760,0.6763,0.6788,0.6769
unseen P1: vs current,0.0000,0.1500,0.1800,0.4300,0.2433
unseen P1: folds,0.0000,11.0000,11.0000,15.0000,9.0000
unseen P1: vs no team,-0.3433,-0.1933,-0.1633,0.0867,-0.1000
unseen both: acc,0.6743,0.6742,0.6752,0.6771,0.6767
unseen both: vs current,0.0000,-0.0033,0.0967,0.2867,0.2433


## 3. Control: permuted team keys

In [5]:
rng = np.random.default_rng(SEED)
X_perm = X.copy()
X_perm[KEYS] = X[KEYS].to_numpy()[rng.permutation(len(X))]
perm = repeated(lambda s: cv_scores(with_team_strength(LGBM, **ENCODERS["R1 + R3"]), X_perm[NUM + KEYS], y, seed=s))
perm_gain = (perm - no_team).mean() * 100
print(f"R1 + R3 with permuted keys: {perm.mean():.4f} vs no team {no_team.mean():.4f} | gain {perm_gain:+.2f} points")

R1 + R3 with permuted keys: 0.6750 vs no team 0.6779 | gain -0.29 points


## 4. Decision

Rule fixed before running: clean no more than 0.1 points below the current encoder; in both unseen scenarios at least
+0.3 points and at least 11/15 folds; permuted keys below +0.2 points.

In [6]:
def passes(name):
    return {
        "clean >= current - 0.1": table.loc[name, "clean: vs current"] >= -0.1,
        "unseen P1 >= +0.3": table.loc[name, "unseen P1: vs current"] >= 0.3,
        "unseen P1 folds >= 11": table.loc[name, "unseen P1: folds"] >= 11,
        "unseen both >= +0.3": table.loc[name, "unseen both: vs current"] >= 0.3,
        "unseen both folds >= 11": table.loc[name, "unseen both: folds"] >= 11,
        "permuted keys < +0.2": perm_gain < 0.2,
    }

for k, v in passes("R1 + R3").items():
    print(f"{'OK ' if v else 'NO '} {k}")
print("ADOPT R1 + R3" if all(passes("R1 + R3").values()) else "DO NOT ADOPT R1 + R3: submissions 22-24 stay as they are")

OK  clean >= current - 0.1
OK  unseen P1 >= +0.3
OK  unseen P1 folds >= 11
NO  unseen both >= +0.3
OK  unseen both folds >= 11
OK  permuted keys < +0.2
DO NOT ADOPT R1 + R3: submissions 22-24 stay as they are


## 5. Summary

- **Checks:** flipping the labels of the 195 battles of one team leaves their pair scores unchanged (change 0); permuted
  keys give -0.29 points against the model without team features, so no gain survives without the real teams.
- **Teams never seen, current encoder (submissions 22 and 24):** -0.34 points (P1 only) and -0.36 (both) against the model
  without team features. This is cleaner than the mask of notebook 21 (-0.23), and worse.
- **R1 + R3:** clean 0.6888 (+0.21 over the current encoder, 8/15 folds), unseen P1 0.6788 (+0.43, 15/15), unseen both
  0.6771 (+0.287, 12/15). Against the model without team features: +1.09 clean, +0.09 and -0.08 on unseen teams.
- **Separately:** R1 alone is the best on clean data (0.6900, +0.33, 10/15) but gains little on unseen teams (+0.18, +0.10);
  R3 alone loses on clean data (-0.22, 2/15). The unseen gain comes from the two together. These are secondary comparisons
  on the same folds, so the best of them is slightly optimistic.
- **Encoder of submission 23 (measured on LightGBM with all features, not on its soft vote):** clean +0.67 against the model
  without team features, -0.10 and -0.12 on unseen teams.

**Decision by the rule fixed before running: not adopted.** Unseen both misses the threshold by 0.013 points (+0.287 vs
+0.3); all the other conditions pass. The miss is far smaller than the noise of the measure, but the rule was fixed in
advance and it is reported as failed. Limits of the test: the model still trains on the battles of the removed teams, and
new species would not be covered.